# 🔥 KURE Fine-tuning 기반 번아웃 분류 모델 v3_ft

## 📋 개요
- **목적**: v3 대비 KURE 상위 레이어 fine-tuning으로 도메인 적응 성능 향상 시도
- **기반**: v3에서 저장한 전처리 데이터 재사용 (stage1/2_train/val_v3.csv)
- **저장 모델**: `stage1_model_v3_ft.pt`, `stage2_model_v3_ft.pt` (기존 v3 보존)

## 🔓 Fine-tuning 전략
- KURE 전체 동결 후 **상위 3개 Transformer 레이어만 해제**
- 차등 learning rate 적용:
  - KURE 해제 레이어: `lr = 1e-5` (매우 작게)
  - 분류기 헤드: `lr = 3e-4` (기존과 동일)
- 임베딩 사전 계산 불가 → 매 step마다 KURE 통과 (학습 느림)

## ⚠️ v3 대비 차이점
| 항목 | v3 | v3_ft |
|------|-----|-------|
| KURE | 완전 동결 | 상위 3레이어 해제 |
| 임베딩 | 사전 계산 | 매 step 계산 |
| 학습 시간 | ~30분 | ~2~3시간 |
| 기대 성능 | 48% | 55~65% (목표) |

---

## 1. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

# 필수 라이브러리
!pip install -q sentence-transformers transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 상수 및 경로 정의

In [ ]:
STAGE1_CATEGORIES = {0: "긍정", 1: "부정"}
STAGE2_CATEGORIES = {
    0: "정서적_고갈",
    1: "좌절_압박",
    2: "부정적_대인관계",
    3: "자기비하"
}

# v3 노트북과 동일한 경로 (이미 저장된 CSV 재사용)
DATA_PATH = "/content/drive/MyDrive/Burnout/dataset"

print("✅ 상수 정의 완료")
print(f"   Stage 1: {list(STAGE1_CATEGORIES.values())}")
print(f"   Stage 2: {list(STAGE2_CATEGORIES.values())}")
print(f"   Data Path: {DATA_PATH}")

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(f"\n📂 데이터 경로: {DATA_PATH}")
required = ['stage1_train_v3.csv', 'stage1_val_v3.csv',
            'stage2_train_v3.csv', 'stage2_val_v3.csv']
for f in required:
    path = f"{DATA_PATH}/{f}"
    exists = os.path.exists(path)
    print(f"   {'✅' if exists else '❌'} {f}")

## 4. 전처리 데이터 로드 및 증강

v3에서 저장한 CSV를 그대로 로드합니다.

In [ ]:
# ============================================================
# v3 전처리 CSV 로드
# ============================================================
s1_train = pd.read_csv(f"{DATA_PATH}/stage1_train_v3.csv")
s1_val   = pd.read_csv(f"{DATA_PATH}/stage1_val_v3.csv")
s2_train = pd.read_csv(f"{DATA_PATH}/stage2_train_v3.csv")
s2_val   = pd.read_csv(f"{DATA_PATH}/stage2_val_v3.csv")

print("✅ 전처리 데이터 로드 완료")
print(f"   S1 Train: {len(s1_train):,}개  |  Val: {len(s1_val):,}개")
print(f"   S2 Train: {len(s2_train):,}개  |  Val: {len(s2_val):,}개")
print(f"\n   S1 분포:\n{s1_train['label'].value_counts().to_string()}")
print(f"\n   S2 분포:")
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_train['label'] == label).sum()
    print(f"     {cat}: {cnt:,}")

In [ ]:
# ============================================================
# 랜덤 혼합 증강 (v3와 동일 로직, 경량 적용)
# fine-tuning은 epoch당 시간이 길어서 augment_ratio=0.3으로 줄임
# ============================================================
def random_mix_augmentation(df, label_col='label', augment_ratio=0.3,
                             min_merge=1, max_merge=3, seed=42):
    rng = np.random.default_rng(seed)
    augmented_rows = []
    for label in sorted(df[label_col].unique()):
        texts = df[df[label_col] == label]['text'].tolist()
        n_generate = max(1, int(len(texts) * augment_ratio))
        for _ in range(n_generate):
            k = rng.integers(min_merge, max_merge + 1)
            k = min(k, len(texts))
            indices = rng.choice(len(texts), size=k, replace=False)
            combined = ' '.join([texts[i] for i in indices])
            augmented_rows.append({'text': combined, label_col: label})
    augmented_df = pd.DataFrame(augmented_rows)
    result = pd.concat([df, augmented_df], ignore_index=True).sample(
        frac=1, random_state=seed).reset_index(drop=True)
    return result


# Stage 2 증강 (Val은 그대로)
s2_train_aug = random_mix_augmentation(s2_train, augment_ratio=0.3)

# Stage 1 증강 (긍정/부정 분리 후)
s1_neg = s1_train[s1_train['label'] == 1].copy()
s1_pos = s1_train[s1_train['label'] == 0].copy()
s1_neg_aug = random_mix_augmentation(s1_neg, augment_ratio=0.3)
s1_pos_aug = random_mix_augmentation(s1_pos, augment_ratio=0.5)
s1_train_aug = pd.concat([s1_neg_aug, s1_pos_aug], ignore_index=True).sample(
    frac=1, random_state=42).reset_index(drop=True)

print("✅ 증강 완료")
print(f"   S1 Train: {len(s1_train):,} → {len(s1_train_aug):,}")
print(f"   S2 Train: {len(s2_train):,} → {len(s2_train_aug):,}")

## 5. KURE 모델 로드 및 Fine-tuning 설정

In [ ]:
print("🔄 KURE 모델 로딩 중...")
st_model = SentenceTransformer('nlpai-lab/KURE-v1')

# 내부 transformer 및 tokenizer 추출
transformer_module = st_model[0]          # sentence_transformers.models.Transformer
tokenizer          = transformer_module.tokenizer
backbone           = transformer_module.auto_model  # HuggingFace 모델

HIDDEN_SIZE = backbone.config.hidden_size
NUM_LAYERS  = len(backbone.encoder.layer)

print(f"✅ KURE 로드 완료")
print(f"   Hidden Size:  {HIDDEN_SIZE}")
print(f"   Encoder 레이어 수: {NUM_LAYERS}")
print(f"   Tokenizer: {tokenizer.__class__.__name__}")

## 6. Fine-tuning 모델 아키텍처

In [ ]:
class FineTunedBurnoutClassifier(nn.Module):
    """
    KURE backbone + 분류기 헤드
    - backbone 상위 N개 레이어만 해제, 나머지 동결
    - Mean Pooling + L2 Normalize → 분류기
    """
    def __init__(self, backbone, num_classes, num_unfreeze_layers=3, dropout=0.5):
        super().__init__()
        self.backbone = backbone
        num_total = len(backbone.encoder.layer)

        # 전체 동결
        for param in self.backbone.parameters():
            param.requires_grad = False

        # 상위 N개 레이어 해제
        for i in range(num_total - num_unfreeze_layers, num_total):
            for param in self.backbone.encoder.layer[i].parameters():
                param.requires_grad = True

        # 상단 LayerNorm 해제 (있는 경우)
        if hasattr(self.backbone, 'LayerNorm'):
            for param in self.backbone.LayerNorm.parameters():
                param.requires_grad = True

        hidden_size = backbone.config.hidden_size  # 1024
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        # 파라미터 수 출력
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"   학습 파라미터: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    def mean_pooling(self, token_embeddings, attention_mask):
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * mask_expanded, 1) / \
               torch.clamp(mask_expanded.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        outputs    = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return self.classifier(embeddings)


class TextDataset(Dataset):
    """텍스트 → 토크나이징 → Dataset"""
    def __init__(self, texts, labels, tokenizer, max_length=256):
        print(f"   토크나이징 중... ({len(texts):,}개)", end=" ")
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
        print("완료")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }


print("✅ 모델/데이터셋 클래스 정의 완료")

## 7. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(-1)
        smooth_targets = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
        smooth_targets = smooth_targets * (1 - self.label_smoothing) + \
                         self.label_smoothing / num_classes
        log_probs = F.log_softmax(inputs, dim=-1)
        probs = torch.exp(log_probs)
        focal_weight = (1 - probs) ** self.gamma
        loss = -focal_weight * smooth_targets * log_probs
        if self.alpha is not None:
            loss = loss * self.alpha[targets].unsqueeze(1)
        return loss.sum(dim=-1).mean()


def compute_class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


print("✅ Loss Functions 정의 완료")

## 8. Fine-tuning 학습 함수

v3 대비 차이점:
- 입력이 임베딩 텐서 → `input_ids` + `attention_mask` 딕셔너리
- **차등 learning rate**: KURE 해제 레이어 `1e-5` / 분류기 헤드 `3e-4`
- batch_size 축소 (GPU 메모리 고려)

In [ ]:
def finetune_model(
    model, train_dataset, val_dataset, train_labels,
    epochs=20, batch_size=16, lr_backbone=1e-5, lr_head=3e-4,
    weight_decay=1e-4, patience=5, min_delta=0.001,
    use_focal_loss=True, label_smoothing=0.1,
    warmup_epochs=2, device='cuda'
):
    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)

    num_classes   = len(np.unique(train_labels))
    class_weights = compute_class_weights(train_labels, num_classes).to(device)

    if use_focal_loss:
        criterion = FocalLoss(gamma=2.0, alpha=class_weights,
                              label_smoothing=label_smoothing)
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights,
                                        label_smoothing=label_smoothing)

    # 차등 learning rate
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params     = list(model.classifier.parameters())
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params,     'lr': lr_head}
    ], weight_decay=weight_decay)

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [],
               'val_acc': [], 'val_f1': [], 'lr_backbone': [], 'lr_head': []}
    best_val_f1, best_state, patience_counter = 0, None, 0

    print(f"\n{'='*65}")
    print(f"🚀 Fine-tuning 시작 (epochs={epochs}, batch={batch_size})")
    print(f"   KURE 레이어 lr={lr_backbone}  |  Head lr={lr_head}")
    print(f"   Class Weights: {class_weights.cpu().numpy().round(3)}")
    print(f"{'='*65}")

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for batch in pbar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            _, pred = torch.max(logits, 1)
            train_total   += labels.size(0)
            train_correct += (pred == labels).sum().item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        scheduler.step()
        train_acc      = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        all_preds, all_labels_list = [], []

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['labels'].to(device)

                logits = model(input_ids, attention_mask)
                loss   = criterion(logits, labels)
                val_loss += loss.item()
                _, pred = torch.max(logits, 1)
                val_total   += labels.size(0)
                val_correct += (pred == labels).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels_list.extend(labels.cpu().numpy())

        val_acc      = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        val_f1       = f1_score(all_labels_list, all_preds, average='weighted')
        lrs          = [g['lr'] for g in optimizer.param_groups]

        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr_backbone'].append(lrs[0])
        history['lr_head'].append(lrs[1])

        print(f"Epoch {epoch+1:2d} | Loss: {avg_train_loss:.4f} | "
              f"Train: {train_acc:.1f}% | Val: {val_acc:.1f}% | "
              f"F1: {val_f1:.4f} | lr_bb: {lrs[0]:.2e}")

        if val_f1 > best_val_f1 + min_delta:
            best_val_f1 = val_f1
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            print(f"         ✅ Best F1: {best_val_f1:.4f} (saved)")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⚠️ Early Stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        print(f"\n✅ Best Model 복원 (F1: {best_val_f1:.4f})")

    return model, history, {'best_f1': best_val_f1, 'best_acc': max(history['val_acc'])}


print("✅ Fine-tuning 학습 함수 정의 완료")

## 9. 데이터셋 토크나이징

In [ ]:
MAX_LENGTH = 256  # 일기 스타일 긴 텍스트 고려

print("🔄 Stage 1 토크나이징...")
s1_train_ds = TextDataset(s1_train_aug['text'], s1_train_aug['label'].values, tokenizer, MAX_LENGTH)
s1_val_ds   = TextDataset(s1_val['text'],       s1_val['label'].values,       tokenizer, MAX_LENGTH)

print("🔄 Stage 2 토크나이징...")
s2_train_ds = TextDataset(s2_train_aug['text'], s2_train_aug['label'].values, tokenizer, MAX_LENGTH)
s2_val_ds   = TextDataset(s2_val['text'],       s2_val['label'].values,       tokenizer, MAX_LENGTH)

print(f"\n✅ 토크나이징 완료")
print(f"   S1 Train: {len(s1_train_ds):,}  |  Val: {len(s1_val_ds):,}")
print(f"   S2 Train: {len(s2_train_ds):,}  |  Val: {len(s2_val_ds):,}")

## 10. Stage 1 Fine-tuning: 긍정 vs 부정

In [ ]:
print("="*65)
print("📊 STAGE 1: 긍정 vs 부정 Fine-tuning")
print("="*65)

stage1_ft = FineTunedBurnoutClassifier(
    backbone=backbone,
    num_classes=2,
    num_unfreeze_layers=3,
    dropout=0.5
).to(device)

S1_FT_CONFIG = {
    'epochs': 15,
    'batch_size': 16,
    'lr_backbone': 1e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'patience': 5,
    'use_focal_loss': False,
    'label_smoothing': 0.1,
    'warmup_epochs': 2
}

stage1_ft, s1_history, s1_best = finetune_model(
    model=stage1_ft,
    train_dataset=s1_train_ds,
    val_dataset=s1_val_ds,
    train_labels=s1_train_aug['label'].values,
    **S1_FT_CONFIG,
    device=device
)

print(f"\n📈 Stage 1 최종 결과:")
print(f"   Best Val Acc: {s1_best['best_acc']:.2f}%")
print(f"   Best Val F1:  {s1_best['best_f1']:.4f}")

## 11. Stage 2 Fine-tuning: 4개 번아웃 카테고리

> ⚠️ Stage 1 학습 후 backbone이 업데이트된 상태입니다.
> Stage 2는 업데이트된 backbone을 이어받아 추가 학습합니다.

In [ ]:
print("="*65)
print("📊 STAGE 2: 4개 번아웃 카테고리 Fine-tuning")
print("="*65)

# Stage 1에서 업데이트된 backbone을 재사용
stage2_ft = FineTunedBurnoutClassifier(
    backbone=backbone,  # Stage 1 fine-tuning 후 backbone
    num_classes=4,
    num_unfreeze_layers=3,
    dropout=0.5
).to(device)

S2_FT_CONFIG = {
    'epochs': 25,
    'batch_size': 16,
    'lr_backbone': 5e-6,   # Stage 2는 더 조심스럽게
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'patience': 6,
    'use_focal_loss': True,
    'label_smoothing': 0.15,
    'warmup_epochs': 3
}

stage2_ft, s2_history, s2_best = finetune_model(
    model=stage2_ft,
    train_dataset=s2_train_ds,
    val_dataset=s2_val_ds,
    train_labels=s2_train_aug['label'].values,
    **S2_FT_CONFIG,
    device=device
)

print(f"\n📈 Stage 2 최종 결과:")
print(f"   Best Val Acc: {s2_best['best_acc']:.2f}%")
print(f"   Best Val F1:  {s2_best['best_f1']:.4f}")

## 12. 결과 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, (history, best, title) in enumerate([
    (s1_history, s1_best, 'Stage 1 (Fine-tuned)'),
    (s2_history, s2_best, 'Stage 2 (Fine-tuned)')
]):
    axes[i, 0].plot(history['train_loss'], label='Train')
    axes[i, 0].plot(history['val_loss'],   label='Val')
    axes[i, 0].set_title(f'{title}: Loss')
    axes[i, 0].legend()

    axes[i, 1].plot(history['train_acc'], label='Train')
    axes[i, 1].plot(history['val_acc'],   label='Val')
    axes[i, 1].set_title(f'{title}: Accuracy (Best: {best["best_acc"]:.1f}%)')
    axes[i, 1].legend()

    axes[i, 2].plot(history['val_f1'])
    axes[i, 2].set_title(f'{title}: Val F1 (Best: {best["best_f1"]:.4f})')

plt.suptitle('KURE Fine-tuning v3_ft', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{DATA_PATH}/training_curves_v3_ft.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ 그래프 저장 완료")

## 13. 상세 평가

In [ ]:
def evaluate_ft_model(model, dataset, labels, categories, stage_name, device='cuda'):
    model.eval()
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids, attention_mask)
            _, pred = torch.max(logits, 1)
            all_preds.extend(pred.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    print(f"\n{'='*60}")
    print(f"📊 {stage_name} 상세 평가")
    print(f"{'='*60}")
    print(classification_report(
        all_labels_list, all_preds,
        target_names=list(categories.values()), digits=4
    ))
    cm = confusion_matrix(all_labels_list, all_preds)
    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=[f"실제_{v}" for v in categories.values()],
        columns=[f"예측_{v}" for v in categories.values()]
    ))
    return all_preds, all_labels_list


evaluate_ft_model(stage1_ft, s1_val_ds, s1_val['label'].values,
                  STAGE1_CATEGORIES, "Stage 1 Fine-tuned", device)

evaluate_ft_model(stage2_ft, s2_val_ds, s2_val['label'].values,
                  STAGE2_CATEGORIES, "Stage 2 Fine-tuned", device)

## 14. 모델 저장 (v3_ft — 기존 v3 보존)

In [ ]:
# ⚠️ v3 모델과 다른 이름으로 저장 (기존 v3 보존)
torch.save({
    'model_state_dict': stage1_ft.state_dict(),
    'backbone_model':   'nlpai-lab/KURE-v1',
    'num_unfreeze_layers': 3,
    'num_classes': 2, 'dropout': 0.5,
    'categories': STAGE1_CATEGORIES,
    'config': S1_FT_CONFIG,
    'best_metrics': s1_best,
    'data_version': 'v3_ft'
}, f"{DATA_PATH}/stage1_model_v3_ft.pt")

torch.save({
    'model_state_dict': stage2_ft.state_dict(),
    'backbone_model':   'nlpai-lab/KURE-v1',
    'num_unfreeze_layers': 3,
    'num_classes': 4, 'dropout': 0.5,
    'categories': STAGE2_CATEGORIES,
    'config': S2_FT_CONFIG,
    'best_metrics': s2_best,
    'data_version': 'v3_ft'
}, f"{DATA_PATH}/stage2_model_v3_ft.pt")

print(f"✅ Stage 1 저장: {DATA_PATH}/stage1_model_v3_ft.pt")
print(f"✅ Stage 2 저장: {DATA_PATH}/stage2_model_v3_ft.pt")
print(f"\n📁 기존 v3 모델 (stage1/2_model_v3.pt) 은 그대로 보존됩니다.")

## 15. 추론 테스트

In [ ]:
def predict_2stage_ft(text, tokenizer, stage1, stage2, max_length=256, device='cuda'):
    stage1.eval(); stage2.eval()
    encoding = tokenizer(
        text, truncation=True, padding=True,
        max_length=max_length, return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        s1_logits = stage1(input_ids, attention_mask)
        s1_probs  = F.softmax(s1_logits, dim=-1)[0]
        s1_pred   = torch.argmax(s1_logits, dim=-1).item()

        result = {
            'text': text,
            'stage1': {
                'category': STAGE1_CATEGORIES[s1_pred],
                'confidence': s1_probs[s1_pred].item(),
                'probs': {STAGE1_CATEGORIES[i]: f"{p.item():.1%}" for i, p in enumerate(s1_probs)}
            },
            'stage2': None
        }

        if s1_pred == 1:  # 부정
            s2_logits = stage2(input_ids, attention_mask)
            s2_probs  = F.softmax(s2_logits, dim=-1)[0]
            s2_pred   = torch.argmax(s2_logits, dim=-1).item()
            result['stage2'] = {
                'category': STAGE2_CATEGORIES[s2_pred],
                'confidence': s2_probs[s2_pred].item(),
                'probs': {STAGE2_CATEGORIES[i]: f"{p.item():.1%}" for i, p in enumerate(s2_probs)}
            }
    return result


test_texts = [
    "오늘도 야근이었다. 집에 오니 아무것도 하기 싫고 그냥 쓰러지고 싶었다.",
    "팀장이 또 내 앞에서 나를 무시했다. 너무 억울하고 화가 난다.",
    "요즘 들어 출근이 너무 싫다. 사람들 얼굴 보기도 싫고 그냥 다 피하고 싶다.",
    "나는 왜 이것밖에 못 할까. 이러니 아무도 날 인정 안 하지.",
    "오늘 발표가 잘 됐다! 팀장님도 칭찬해 주셔서 기분이 좋았다.",
    "잠을 못 잤더니 온종일 멍했다. 아무것도 집중이 안 되고 너무 지쳤다."
]

print("🧪 일기 스타일 텍스트 테스트 (Fine-tuned)")
for text in test_texts:
    r  = predict_2stage_ft(text, tokenizer, stage1_ft, stage2_ft, device=device)
    s1 = r['stage1']
    s2 = r['stage2']
    print(f"\n📝 {r['text'][:50]}...")
    print(f"   Stage 1: {s1['category']} ({s1['confidence']:.1%})")
    if s2:
        print(f"   Stage 2: {s2['category']} ({s2['confidence']:.1%})")
        print(f"   확률: {s2['probs']}")

## 16. 최종 요약 (v3 vs v3_ft 비교)

In [ ]:
print("="*70)
print("📋 Fine-tuning 결과 요약 (v3_ft)")
print("="*70)

print("\n🎯 성능 비교")
print("-"*70)
print(f"  {'':30s}  {'v3 (Frozen)':>15s}  {'v3_ft (Fine-tuned)':>18s}")
print(f"  {'Stage 1 F1':30s}  {'0.9877':>15s}  {s1_best['best_f1']:>18.4f}")
print(f"  {'Stage 2 F1':30s}  {'0.4811':>15s}  {s2_best['best_f1']:>18.4f}")
print(f"  {'Stage 2 Acc':30s}  {'48.1%':>15s}  {s2_best['best_acc']:>17.2f}%")

improvement = s2_best['best_f1'] - 0.4811
print(f"\n  Stage 2 F1 변화: {'+' if improvement >= 0 else ''}{improvement:.4f}")
if improvement > 0.02:
    print("  ✅ Fine-tuning 효과 있음")
elif improvement > 0:
    print("  ↔ 미미한 향상")
else:
    print("  ❌ Fine-tuning 효과 없음 (v3 사용 권장)")

print("\n🔧 Fine-tuning 설정")
print("-"*70)
print("  - KURE 해제 레이어: 상위 3개")
print(f"  - S1 lr_backbone={S1_FT_CONFIG['lr_backbone']}  lr_head={S1_FT_CONFIG['lr_head']}")
print(f"  - S2 lr_backbone={S2_FT_CONFIG['lr_backbone']}  lr_head={S2_FT_CONFIG['lr_head']}")
print("\n" + "="*70)